# ⚡ Kilat: Supervised Fine-Tuning (SFT) Alignment

```text
  ██╗  ██╗██╗██╗      █████╗ ████████╗
  ██║ ██╔╝██║██║     ██╔══██╗╚══██╔══╝
  █████╔╝ ██║██║     ███████║   ██║   
  ██╔═██╗ ██║██║     ██╔══██║   ██║   
  ██║  ██╗██║███████╗██║  ██║   ██║   
  ╚═╝  ╚═╝╚═╝╚══════╝╚═╝  ╚═╝   ╚═╝   
  ────────────────────────────────────────────────────────────
  ⚡ KILAT | Kernelized Lightweight Attention Transformer (v2.0)
  Architecture: Hybrid Sub-Quadratic Global Decay + Latent MLA
  ────────────────────────────────────────────────────────────
```

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Airukua/Kilat/blob/main/notebooks/kilat_sft_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Airukua%2FKilat-181717?style=flat-square&logo=github)](https://github.com/Airukua/Kilat)
[![Hugging Face](https://img.shields.io/badge/🤗%20HuggingFace-AiRukua%2F150M--Kilat--SMALL--10B-FFD21E?style=flat-square)](https://huggingface.co/AiRukua/150M-Kilat-SMALL-10B)

Welcome to the **Kilat SFT Alignment** guide!
Supervised Fine-Tuning (SFT) transforms a raw base language model into an instruction-following assistant.

Key Features of Kilat SFT:
1. **Prompt Loss Masking (`-100`)**: Tokens belonging to the prompt or user question are masked with `-100`, ensuring the loss gradient only trains the assistant's response.
2. **Non-Reentrant Activation Checkpointing**: Dramatically reduces VRAM consumption during backward pass without PyTorch autograd graph leaks.
3. **`KilatSFTTrainer`**: Dedicated high-throughput alignment trainer with cosine warmup and gradient accumulation.

## 1. Environment Setup & Kilat Installation

In [ ]:
!rm -rf Kilat
!git clone https://github.com/Airukua/Kilat.git
%cd Kilat
!pip install -q -e ".[integrations]"

from kilat.utils.banner import print_kilat_banner
print_kilat_banner()

## 2. Load Base Model & Tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "AiRukua/150M-Kilat-SMALL-10B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
if torch.cuda.is_available():
    model = model.cuda()

print("✅ Base Model ready for SFT!")

## 3. Prompt Loss Masking Invariant (`labels = -100`)
In standard causal language modeling, every token is predicted. However, in SFT, we do NOT want the model to learn to predict the user's prompt. We must mask prompt positions with `-100`.

In [ ]:
from kilat.data.sft_dataset import SFTDataset

# Sample instruction dataset
sample_data = [
    {
        "prompt": "User: What is mixture of experts?\nAssistant:",
        "response": " Mixture of Experts (MoE) activates only a subset of network parameters per token, enabling massive parameter capacity with low compute cost."
    },
    {
        "prompt": "User: Why is global decay useful?\nAssistant:",
        "response": " Global decay scales sequence modeling with O(1) recurrent state updates, eliminating quadratic attention bottlenecks."
    }
]

# Verify token masking
prompt_ids = tokenizer.encode(sample_data[0]["prompt"])
response_ids = tokenizer.encode(sample_data[0]["response"])

full_ids = prompt_ids + response_ids
labels = [-100] * len(prompt_ids) + response_ids

print(f"Total Tokens:      {len(full_ids)}")
print(f"Masked (Prompt):   {len(prompt_ids)} tokens set to -100")
print(f"Trained (Response): {len(response_ids)} tokens actively trained")

## 4. Configure Kilat SFT Trainer
Configure `KilatSFTTrainer` with non-reentrant activation checkpointing for maximum memory efficiency.

In [ ]:
from kilat.engine.sft_trainer import KilatSFTTrainer
from kilat.engine.args import TrainingArguments

sft_args = TrainingArguments(
    output_dir="./checkpoints/kilat_sft_aligned",
    learning_rate=2e-5,
    min_lr=2e-6,
    warmup_steps=20,
    max_steps=100,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    mixed_precision="fp16" if not torch.cuda.is_bf16_supported() else "bf16",
    gradient_checkpointing=True,
    save_steps=50,
    logging_steps=10,
    seed=42
)

print("⚡ SFT Alignment Trainer Configured!")

## 5. Interactive Chat Inference
Test the model with prompt-response generation.

In [ ]:
test_prompt = "User: Tell me about the Kilat framework.\nAssistant:"
inputs = tokenizer(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id
    )

print("⚡ [PROMPT]:\n", test_prompt)
print("⚡ [RESPONSE]:\n", tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))